## Sampling for the golden dataset

Builds samples of growing difficulty from the clustering result (`corpus/chunks/clusters.json`):

| level | sample | chunks drawn from |
|---|---|---|
| 1 | single chunk | one cluster |
| 2 | a few chunks | the same cluster |
| 3 | chunks from two **similar** clusters | neighbouring centroids |
| 4 | chunks from two **different** clusters | far-apart centroids |

Excluded: noise chunks (cluster -1) and the clusters listed in `EXCLUDE_CLUSTERS`.
A chunk is used at most once, and no two chunks of one stream with overlapping time spans are used.

In [1]:
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np

# ---- parameters ----
SEED = 42
LEVEL_COUNTS = {1: 20, 2: 15, 3: 15, 4: 10}      # samples per level
EXCLUDE_CLUSTERS = {7}                           # meaningless clusters (general chat / wrap-up talk)
CHUNKS_L2 = 3                                    # chunks per level-2 sample
K_NEIGHBOURS = 3                                 # "similar" = top-K nearest clusters, "different" = K farthest
MAX_PER_CLUSTER = 2                              # a cluster may appear in at most 2 samples of level 3 / level 4

ROOT = Path.cwd().resolve()
while not (ROOT / "corpus" / "chunks").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CHUNKS_PATH = ROOT / "corpus" / "chunks" / "time_chunks.json"
CLUSTERS_PATH = ROOT / "corpus" / "chunks" / "clusters.json"
OUT_PATH = ROOT / "corpus" / "samples" / "samples.json"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
V = np.load(CHUNKS_PATH.with_suffix(".npy"))
cl = json.loads(CLUSTERS_PATH.read_text(encoding="utf-8"))

ids = [c["id"] for c in chunks]
assert list(cl["labels"]) == ids, "clusters.json does not match time_chunks.json - re-run the clustering notebook"
labels = np.array([cl["labels"][i] for i in ids])

# clusters we may sample from
valid = [k for k in cl["centroid_similarity"]["cluster_ids"] if k not in EXCLUDE_CLUSTERS]
sim_ids = cl["centroid_similarity"]["cluster_ids"]
SIM = np.array(cl["centroid_similarity"]["matrix"])
sim = {(a, b): SIM[sim_ids.index(a), sim_ids.index(b)] for a in valid for b in valid if a != b}

# members of every valid cluster, and the "typical" half (closest to the centroid)
members, typical = {}, {}
for k in valid:
    idx = np.where(labels == k)[0]
    cen = V[idx].mean(axis=0)
    cen /= np.linalg.norm(cen)
    order = idx[np.argsort(-(V[idx] @ cen))]
    members[k] = [int(i) for i in order]
    typical[k] = members[k][: max(1, len(order) // 2)]

print(f"{len(valid)} usable clusters, {sum(len(m) for m in members.values())} usable chunks "
      f"(excluded: clusters {sorted(EXCLUDE_CLUSTERS)} and {int((labels < 0).sum())} noise chunks)")

21 usable clusters, 518 usable chunks (excluded: clusters [7] and 245 noise chunks)


In [2]:
rng = random.Random(SEED)


def overlap(i, j):
    a, b = chunks[i], chunks[j]
    return a["youtubeId"] == b["youtubeId"] and a["start"] < b["end"] and b["start"] < a["end"]


used = []          # chunks already placed in finished samples


def pick(k, taken, typical_only, prefer_new_stream=False):
    """A random free chunk of cluster k that overlaps nothing already used or in `taken`; None if there is none."""
    for pool in ([typical[k], members[k]] if typical_only else [members[k]]):
        cand = [i for i in pool if not any(overlap(i, j) for j in used + taken)]
        rng.shuffle(cand)
        if prefer_new_stream:
            seen = {chunks[j]["youtubeId"] for j in taken}
            cand.sort(key=lambda i: chunks[i]["youtubeId"] in seen)    # stable: new streams first
        if cand:
            return cand[0]
    return None


samples = []


def add(level, kind, chunk_idx, extra=None):
    used.extend(chunk_idx)
    n = sum(1 for s in samples if s["level"] == level) + 1
    samples.append({
        "sample_id": f"L{level}-{n:02d}",
        "level": level,
        "type": kind,
        "chunk_ids": [ids[i] for i in chunk_idx],
        "clusters": [int(labels[i]) for i in chunk_idx],
        "spans": [{"youtubeId": chunks[i]["youtubeId"], "start": chunks[i]["start"], "end": chunks[i]["end"],
                   "url": f"{chunks[i]['url']}?t={chunks[i]['start']}"} for i in chunk_idx],
        "tokens": sum(chunks[i]["tokens"] for i in chunk_idx),
        **(extra or {}),
    })


def cluster_cycle():
    """Shuffled clusters, repeated, so samples spread evenly over the clusters."""
    while True:
        order = valid[:]
        rng.shuffle(order)
        yield from order


# ---- level 1: one chunk, typical for its cluster ----
for k in cluster_cycle():
    if sum(1 for s in samples if s["level"] == 1) >= LEVEL_COUNTS[1]:
        break
    i = pick(k, [], typical_only=True)
    if i is not None:
        add(1, "single_chunk", [i])

# ---- level 2: a few chunks of one cluster, preferably from different streams ----
tries = 0
for k in cluster_cycle():
    tries += 1
    if sum(1 for s in samples if s["level"] == 2) >= LEVEL_COUNTS[2] or tries > 20 * len(valid):
        break
    taken = []
    for _ in range(CHUNKS_L2):
        i = pick(k, taken, typical_only=True, prefer_new_stream=True)
        if i is None:
            break
        taken.append(i)
    if len(taken) == CHUNKS_L2:
        add(2, "same_cluster", taken, {"n_streams": len({chunks[i]["youtubeId"] for i in taken})})


# ---- levels 3 and 4: one chunk from each of two clusters ----
def cluster_pairs(similar):
    pairs = set()
    for a in valid:
        others = sorted((b for b in valid if b != a), key=lambda b: sim[(a, b)], reverse=similar)
        pairs.update(tuple(sorted((a, b))) for b in others[:K_NEIGHBOURS])
    pairs = sorted(pairs)
    rng.shuffle(pairs)
    return pairs


similar_pairs, different_pairs = cluster_pairs(True), cluster_pairs(False)
assert not set(similar_pairs) & set(different_pairs), "a cluster pair is both similar and different - lower K_NEIGHBOURS"

for level, kind, pairs in [(3, "similar_clusters", similar_pairs), (4, "different_clusters", different_pairs)]:
    usage = Counter()
    for a, b in pairs:
        if sum(1 for s in samples if s["level"] == level) >= LEVEL_COUNTS[level]:
            break
        if usage[a] >= MAX_PER_CLUSTER or usage[b] >= MAX_PER_CLUSTER:
            continue
        ia = pick(a, [], typical_only=False)
        ib = pick(b, [ia] if ia is not None else [], typical_only=False)
        if ia is None or ib is None:
            continue
        add(level, kind, [ia, ib], {"centroid_similarity": round(float(sim[(a, b)]), 4)})
        usage[a] += 1
        usage[b] += 1

print({lvl: sum(1 for s in samples if s["level"] == lvl) for lvl in LEVEL_COUNTS})

{1: 20, 2: 15, 3: 15, 4: 10}


In [3]:
# ---- Checks ----
by_level = Counter(s["level"] for s in samples)
for lvl, want in LEVEL_COUNTS.items():
    assert by_level[lvl] == want, f"level {lvl}: got {by_level[lvl]} of {want} samples - relax the constraints"

all_idx = [ids.index(cid) for s in samples for cid in s["chunk_ids"]]
assert len(all_idx) == len(set(all_idx)), "a chunk is used twice"
assert not any(overlap(i, j) for n, i in enumerate(all_idx) for j in all_idx[:n]), "overlapping spans of one stream"
assert all(c not in EXCLUDE_CLUSTERS and c >= 0 for s in samples for c in s["clusters"]), "excluded cluster or noise used"

for lvl in (3, 4):
    sims = [s["centroid_similarity"] for s in samples if s["level"] == lvl]
    print(f"level {lvl}: centroid similarity of the cluster pairs  min {min(sims):.2f}  mean {np.mean(sims):.2f}  max {max(sims):.2f}")
assert (np.mean([s["centroid_similarity"] for s in samples if s["level"] == 3])
        > np.mean([s["centroid_similarity"] for s in samples if s["level"] == 4])), "levels 3 and 4 are not ordered by similarity"

print("clusters used per level:", {lvl: len({c for s in samples if s["level"] == lvl for c in s["clusters"]}) for lvl in LEVEL_COUNTS})
print("level 2, streams per sample:", dict(Counter(s["n_streams"] for s in samples if s["level"] == 2)))
print("tokens per sample (min/mean/max):", min(s["tokens"] for s in samples), int(np.mean([s["tokens"] for s in samples])), max(s["tokens"] for s in samples))

OUT_PATH.write_text(json.dumps({
    "config": {"seed": SEED, "level_counts": LEVEL_COUNTS, "exclude_clusters": sorted(EXCLUDE_CLUSTERS),
               "chunks_level2": CHUNKS_L2, "k_neighbours": K_NEIGHBOURS, "max_per_cluster": MAX_PER_CLUSTER,
               "clusters_file": CLUSTERS_PATH.name},
    "samples": samples,
}, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"{len(samples)} samples -> {OUT_PATH}")

level 3: centroid similarity of the cluster pairs  min 0.84  mean 0.87  max 0.90
level 4: centroid similarity of the cluster pairs  min 0.76  mean 0.78  max 0.80
clusters used per level: {1: 20, 2: 15, 3: 18, 4: 13}
level 2, streams per sample: {3: 2, 1: 8, 2: 5}
tokens per sample (min/mean/max): 770 1555 2536
60 samples -> S:\ai_projects\demo_26_RAG-vs-agent\corpus\samples\samples.json


In [5]:
# ---- One example per level ----
by_id = {c["id"]: c for c in chunks}
for lvl in LEVEL_COUNTS:
    s = next(s for s in samples if s["level"] == lvl)
    print(f"\n##### {s['sample_id']} ({s['type']}) clusters={s['clusters']} tokens={s['tokens']}")
    for cid in s["chunk_ids"]:
        c = by_id[cid]
        print(f"  [{cid}] {c['url']}?t={c['start']}\n      {c['text']}...")


##### L1-01 (single_chunk) clusters=[20] tokens=805
  [ROs1CGNbXdo#02:13:57] https://youtu.be/ROs1CGNbXdo?t=8037
      не сказав, що це дуже складна якась штука, але ти її відчуваєш, коли ти попрацював і там. Просто в мене є досвід. Я був і в аутсорці, і я в продукті був. І я відчуваю цю різницю. І причому дуже сильно відчуваю різницю відношення і розробників, і всього стафу повністю. Це дуже кардинальна рі різниця. Іноді це досить важко чомусь зрозуміти, але коли ти і там побував, ти розумієш прям дуже сильно. І коли наша компанія наймає аутсорс якихось сотрудників, мене начинає десь там на покалувать трошечки. Я такий: "На якого зачем? Не надо нам". да. То що я прям відчуваю знову оце дудуновение. Я понимаю, як вони ра я знаю, як вони работають, а там ще й специфіка тої країни, звідкиля вони. Я прям дуже хорошо розбираюся, як і що вони делають. І да, тут не прийнято, конечно, знаєш, типа це все там якісь росследование делать, хто що робить. це не моя задача абсолютно. Але я просто, 